# 🏆 BTK Datathon 2026 — Career Success Score (v4 Ultimate)

**Yarışma Metriği:** MSE (Mean Squared Error) — düşük = iyi  
**Optimizasyon:** RMSE (MSE'yi minimize etmek RMSE'yi minimize etmekle eşdeğer)  
**Donanım:** NVIDIA CUDA GPU

## v4'te Neler Yeni?

v3'e ek olarak (eleştirel düşünmeyle değişen mimari):

1. **🧠 Multilingual Sentence-BERT Embeddings** — Bag-of-words yerine **semantik** anlam
2. **🤖 7 Çeşit Base Model** — Sadece ağaç değil, **4 farklı algoritma türü**:
   - Tree (XGB, LGB, CAT, HGB)
   - Neural (MLP)
   - Linear+Polynomial (Ridge with PolynomialFeatures)
   - Distance-based (KNN)
3. **📊 10-Fold CV** — 5'ten 10'a (daha güvenilir OOF)
4. **🔬 100+ Optuna Trial** — Daha derinlemesine arama
5. **🎯 Cross-Categorical Target Encoding** — `department × target_role` gibi
6. **🔄 Pseudo-Labeling A/B Test** — Sadece kanıtlanmışsa kullan
7. **🏆 2-Seviyeli Stacking** — Ridge meta + SLSQP weight, en iyisi seçilir

**Beklenen:** v1 89.54 (MSE) → v4 ~72-78 (MSE) — ilk 10 hedefi

In [ ]:
# ── Paket kurulumu (Colab'da çalışacak) ──────────────────────────────────────
!pip install sentence-transformers -q 2>&1 | tail -1
!pip install xgboost lightgbm catboost optuna -q 2>&1 | tail -1

In [ ]:
# ── Kütüphaneler ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings, gc, time, os
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from scipy.optimize import minimize

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

import optuna
from optuna.pruners import SuccessiveHalvingPruner, MedianPruner
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED       = 42
N_FOLDS    = 10           # v3: 5 → v4: 10
N_SEEDS    = 3
TARGET_COL = 'career_success_score'

np.random.seed(SEED)
print('Kütüphaneler hazır ✓')

In [ ]:
# ── GPU Kontrol ──────────────────────────────────────────────────────────────
GPU_AVAILABLE = False; LGB_GPU = False; CAT_GPU = False; BERT_DEVICE = 'cpu'

test_X = np.random.rand(100, 5).astype(np.float32)
test_y = np.random.rand(100).astype(np.float32)

try:
    m = xgb.XGBRegressor(n_estimators=10, device='cuda', tree_method='hist')
    m.fit(test_X, test_y)
    GPU_AVAILABLE = True; print('✓ XGBoost GPU')
except: print('✗ XGBoost CPU')

try:
    m = lgb.LGBMRegressor(n_estimators=10, device='gpu', verbose=-1)
    m.fit(test_X, test_y)
    LGB_GPU = True; print('✓ LightGBM GPU')
except: print('✗ LightGBM CPU')

try:
    m = CatBoostRegressor(iterations=10, task_type='GPU', devices='0', verbose=0)
    m.fit(test_X, test_y)
    CAT_GPU = True; print('✓ CatBoost GPU')
except: print('✗ CatBoost CPU')

try:
    import torch
    if torch.cuda.is_available():
        BERT_DEVICE = 'cuda'
        print(f'✓ PyTorch CUDA: {torch.cuda.get_device_name(0)}')
    else:
        print('✗ PyTorch CPU (BERT yavaş çalışacak)')
except:
    print('✗ PyTorch yok')

xgb_gpu = lambda: {'device':'cuda','tree_method':'hist'} if GPU_AVAILABLE else {'tree_method':'hist','n_jobs':-1}
lgb_gpu = lambda: {'device':'gpu'} if LGB_GPU else {'n_jobs':-1}
cat_gpu = lambda: {'task_type':'GPU','devices':'0'} if CAT_GPU else {}

## 1. Veri Yükleme

In [ ]:
TRAIN_PATH  = 'train.csv'
TEST_PATH   = 'test_x.csv'
SAMPLE_PATH = 'sample_submission.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_PATH)

y         = train[TARGET_COL].values
test_ids  = test['student_id'].values
y_bin     = pd.qcut(y, q=10, labels=False, duplicates='drop')
skf       = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLD_SPLITS = list(skf.split(train, y_bin))

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Folds: {N_FOLDS}')

## 2. 🧠 Sentence-BERT Embeddings (v4 yeni — en büyük tek yenilik)

**Multilingual MiniLM** modeli kullanılır — TF-IDF'in yapamadığını yapar: aynı anlamı taşıyan farklı kelimeleri ("exceptional" vs "outstanding") **yakın vektörlere** map eder.

Eğer internet erişimi yoksa otomatik olarak TF-IDF only fallback'e geçer.

In [ ]:
BERT_OK = False
bert_embeddings = None

try:
    from sentence_transformers import SentenceTransformer
    
    print('Sentence-BERT modeli yükleniyor...')
    bert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=BERT_DEVICE)
    
    all_texts = pd.concat([
        train['mentor_feedback_text'],
        test['mentor_feedback_text']
    ]).fillna('').tolist()
    
    print(f'Encoding {len(all_texts)} texts on {BERT_DEVICE}...')
    t0 = time.time()
    bert_embeddings = bert_model.encode(
        all_texts, batch_size=128, show_progress_bar=True,
        convert_to_numpy=True, device=BERT_DEVICE
    )
    print(f'BERT done: {bert_embeddings.shape} in {time.time()-t0:.1f}s')
    BERT_OK = True
    
    # PCA reduction
    pca = PCA(n_components=64, random_state=SEED)
    bert_reduced = pca.fit_transform(bert_embeddings)
    print(f'PCA reduced to 64-dim, explained var: {pca.explained_variance_ratio_.sum():.3f}')
    
    bert_cols = [f'bert_{i}' for i in range(64)]
    train_bert = pd.DataFrame(bert_reduced[:len(train)], columns=bert_cols)
    test_bert  = pd.DataFrame(bert_reduced[len(train):], columns=bert_cols)
    
    # Temizlik
    del bert_model, bert_embeddings
    gc.collect()
    if BERT_DEVICE == 'cuda':
        import torch; torch.cuda.empty_cache()
    print('✓ Sentence-BERT embeddings hazır')
    
except Exception as e:
    print(f'⚠ BERT kullanılamıyor: {str(e)[:100]}')
    print('TF-IDF only modunda devam edilecek')
    train_bert = pd.DataFrame()
    test_bert  = pd.DataFrame()

## 3. Geleneksel NLP — TF-IDF (BERT'e ek)

BERT semantik, TF-IDF leksik — ikisi farklı sinyalleri yakalar, beraber kullanmak en iyisi.

In [ ]:
def build_tfidf_features(train_df, test_df, word_n=60, char_n=40):
    all_texts = pd.concat([
        train_df['mentor_feedback_text'],
        test_df['mentor_feedback_text']
    ]).fillna('').reset_index(drop=True)

    word_tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=3,
                                  sublinear_tf=True, analyzer='word')
    word_m = word_tfidf.fit_transform(all_texts)
    word_feats = TruncatedSVD(n_components=word_n, random_state=SEED).fit_transform(word_m)

    char_tfidf = TfidfVectorizer(max_features=3000, ngram_range=(3, 5), min_df=3,
                                  sublinear_tf=True, analyzer='char_wb')
    char_m = char_tfidf.fit_transform(all_texts)
    char_feats = TruncatedSVD(n_components=char_n, random_state=SEED).fit_transform(char_m)

    cols = [f'nlp_w_{i}' for i in range(word_n)] + [f'nlp_c_{i}' for i in range(char_n)]
    nlp = pd.DataFrame(np.hstack([word_feats, char_feats]), columns=cols)
    nlp['text_len']   = all_texts.str.len().values
    nlp['word_count'] = all_texts.str.split().str.len().values

    return (nlp.iloc[:len(train_df)].reset_index(drop=True),
            nlp.iloc[len(train_df):].reset_index(drop=True))

train_nlp, test_nlp = build_tfidf_features(train, test)
print(f'TF-IDF features: {train_nlp.shape}')

## 4. 🎯 Role-Skill Engineering (v3'te kanıtlandı)

In [ ]:
ROLE_PROFILES = {
    'Cloud Engineer':        {'cloud_score':3, 'devops_score':2, 'problem_solving_score':1},
    'DevOps Engineer':       {'devops_score':3, 'cloud_score':2, 'problem_solving_score':1},
    'MLOps Engineer':        {'devops_score':2, 'machine_learning_score':2, 'cloud_score':1, 'problem_solving_score':1},
    'Frontend Developer':    {'frontend_score':3, 'coding_score':2, 'problem_solving_score':1},
    'Backend Developer':     {'backend_score':3, 'coding_score':2, 'sql_score':1, 'problem_solving_score':1},
    'Software Developer':    {'coding_score':2, 'problem_solving_score':2, 'data_structures_score':2, 'backend_score':1, 'frontend_score':1},
    'Data Scientist':        {'machine_learning_score':3, 'sql_score':2, 'problem_solving_score':2, 'data_structures_score':1},
    'AI Engineer':           {'machine_learning_score':3, 'coding_score':2, 'problem_solving_score':1, 'data_structures_score':1},
    'Data Analyst':          {'sql_score':3, 'data_structures_score':1, 'problem_solving_score':1},
    'Product Analyst':       {'sql_score':2, 'problem_solving_score':2},
    'Cybersecurity Analyst': {'devops_score':2, 'problem_solving_score':2, 'coding_score':1},
}
PRIMARY_SKILL = {r: max(w.items(), key=lambda x: x[1])[0] for r, w in ROLE_PROFILES.items()}

def add_role_features(df):
    df = df.copy()
    df['matched_skill'] = 0.0
    for role, skill in PRIMARY_SKILL.items():
        m = df['target_role'] == role
        df.loc[m, 'matched_skill'] = df.loc[m, skill].values
    df['role_composite'] = 0.0
    for role, w in ROLE_PROFILES.items():
        m = df['target_role'] == role
        if m.sum() == 0: continue
        tot = sum(w.values())
        df.loc[m, 'role_composite'] = (sum(df.loc[m, sk]*wt for sk, wt in w.items()) / tot).values
    return df

train = add_role_features(train)
test  = add_role_features(test)
print('✓ Role-skill features eklendi')

## 5. 🆕 Cross-Categorical Target Encoding

Standart target encoding'e ek olarak `department × target_role`, `university_tier × target_role` gibi **çapraz** kombinasyonlar da encode edilir. Bu, modele daha ince segment bilgisi verir.

In [ ]:
def kfold_te(train_df, test_df, col, target, splits, smoothing=20):
    """Sızıntısız K-fold target encoding (tek sütun veya birleşik)."""
    gm = train_df[target].mean()
    oof = np.zeros(len(train_df))
    for tr_idx, val_idx in splits:
        tr = train_df.iloc[tr_idx]
        agg = tr.groupby(col)[target].agg(['mean', 'count'])
        agg['s'] = (agg['mean']*agg['count'] + gm*smoothing)/(agg['count']+smoothing)
        val = train_df.iloc[val_idx]
        oof[val_idx] = val[col].map(agg['s']).fillna(gm).values
    agg = train_df.groupby(col)[target].agg(['mean', 'count'])
    agg['s'] = (agg['mean']*agg['count'] + gm*smoothing)/(agg['count']+smoothing)
    te = test_df[col].map(agg['s']).fillna(gm).values
    return oof, te

# Tek değişkenli
train_with_y = train.assign(**{TARGET_COL: y})
te_train, te_test = {}, {}

single_cols = ['department','target_role','hobby','preferred_social_media_platform','university_tier']
for col in single_cols:
    tr_e, te_e = kfold_te(train_with_y, test, col, TARGET_COL, FOLD_SPLITS, smoothing=20)
    te_train[f'te_{col}'] = tr_e
    te_test[f'te_{col}']  = te_e

# Çapraz kategorik (v4 yeni)
cross_pairs = [
    ('department','target_role'),
    ('university_tier','target_role'),
    ('target_role','hobby'),
]
train_cross = train_with_y.copy()
test_cross  = test.copy()
for c1, c2 in cross_pairs:
    combo = f'{c1}_x_{c2}'
    train_cross[combo] = train_cross[c1].astype(str) + '|' + train_cross[c2].astype(str)
    test_cross[combo]  = test_cross[c1].astype(str) + '|' + test_cross[c2].astype(str)
    tr_e, te_e = kfold_te(train_cross, test_cross, combo, TARGET_COL, FOLD_SPLITS, smoothing=30)
    te_train[f'te_{combo}'] = tr_e
    te_test[f'te_{combo}']  = te_e
    print(f'  te_{combo}: corr={np.corrcoef(tr_e, y)[0,1]:.3f}')

train_te = pd.DataFrame(te_train)
test_te  = pd.DataFrame(te_test)
print(f'Toplam TE feature: {train_te.shape[1]}')

## 6. Feature Engineering — Tam Pipeline

In [ ]:
TIER_MAP  = {'Tier 1':4, 'Tier 2':3, 'Tier 3':2, 'Tier 4':1}
TECH_COLS = ['coding_score','problem_solving_score','data_structures_score',
             'sql_score','machine_learning_score','backend_score','frontend_score',
             'cloud_score','devops_score']
SOFT_COLS = ['communication_score','teamwork_score','leadership_score','presentation_score']
CAT_COLS  = ['department','target_role','hobby','preferred_social_media_platform']

def build_features(train_df, test_df):
    train_df = train_df.copy(); test_df = test_df.copy()

    fill_cols = ['english_exam_score','internship_duration_months','portfolio_score',
                 'github_avg_stars','open_source_contribution_count',
                 'linkedin_profile_score','hr_interview_score']
    medians = {c: train_df[c].median() for c in fill_cols}
    for c in fill_cols:
        train_df[c] = train_df[c].fillna(medians[c])
        test_df[c]  = test_df[c].fillna(medians[c])

    def derive(df):
        df = df.copy()
        # Teknik aggregateler
        df['tech_mean']  = df[TECH_COLS].mean(axis=1)
        df['tech_max']   = df[TECH_COLS].max(axis=1)
        df['tech_min']   = df[TECH_COLS].min(axis=1)
        df['tech_std']   = df[TECH_COLS].std(axis=1)
        df['tech_sum']   = df[TECH_COLS].sum(axis=1)
        df['tech_range'] = df['tech_max'] - df['tech_min']
        df['tech_top3']  = df[TECH_COLS].apply(lambda r: r.nlargest(3).mean(), axis=1)
        df['tech_bot3']  = df[TECH_COLS].apply(lambda r: r.nsmallest(3).mean(), axis=1)
        # Soft
        df['soft_mean'] = df[SOFT_COLS].mean(axis=1)
        df['soft_sum']  = df[SOFT_COLS].sum(axis=1)
        df['soft_std']  = df[SOFT_COLS].std(axis=1)
        df['soft_min']  = df[SOFT_COLS].min(axis=1)
        # Role interactions (kanıtlanmış)
        df['matched_x_proj']        = df['matched_skill'] * df['project_quality_score']
        df['matched_x_interview']   = df['matched_skill'] * df['technical_interview_score']
        df['role_comp_x_proj']      = df['role_composite'] * df['project_quality_score']
        df['role_comp_x_interview'] = df['role_composite'] * df['technical_interview_score']
        df['matched_minus_tech']    = df['matched_skill'] - df['tech_mean']
        df['role_comp_minus_avg']   = df['role_composite'] - df['tech_mean']
        df['matched_x_role_comp']   = df['matched_skill'] * df['role_composite'] / 100
        # Mülakat
        df['interview_composite'] = df['technical_interview_score']*0.6 + df['hr_interview_score']*0.4
        df['interview_diff']      = df['technical_interview_score'] - df['hr_interview_score']
        # Deneyim
        df['total_projects']  = df['real_client_project_count'] + df['freelance_project_count']
        df['experience_score']= (df['internship_count']*3 + df['real_client_project_count']*2 + 
                                  df['freelance_project_count'])
        df['exp_per_year']    = df['experience_score'] / (df['age'] - 18).clip(lower=1)
        # GitHub
        df['github_score']    = (df['github_repo_count'] * np.log1p(df['github_avg_stars']+1) +
                                  df['open_source_contribution_count'])
        df['oss_ratio']       = df['open_source_contribution_count'] / (df['github_repo_count']+1)
        # Profil
        df['profile_composite']= (df['portfolio_score']+df['linkedin_profile_score']+df['cv_quality_score'])/3
        df['profile_min']      = df[['portfolio_score','linkedin_profile_score','cv_quality_score']].min(axis=1)
        df['profile_max']      = df[['portfolio_score','linkedin_profile_score','cv_quality_score']].max(axis=1)
        # Hackathon & başvuru
        df['hackathon_eff']   = np.where(df['hackathon_count']>0, df['hackathon_awards']/df['hackathon_count'], 0)
        df['app_success_rate']= df['interviews_attended'] / (df['applications_sent']+1)
        # Üni & eğitim
        df['university_tier_num'] = df['university_tier'].map(TIER_MAP)
        df['years_to_grad'] = df['graduation_year'] - df['application_year']
        df['cert_per_internship'] = df['certification_count'] / (df['internship_count']+1)
        df['cgpa_x_attendance']   = df['cgpa'] * df['attendance_rate']
        df['failed_penalty']      = df['failed_courses_count'] / (df['cgpa']+0.01)
        # Genel interactions
        df['soft_x_technical']    = df['soft_mean'] * df['technical_interview_score']
        df['proj_x_interview']    = df['project_quality_score'] * df['technical_interview_score']
        df['proj_x_portfolio']    = df['project_quality_score'] * df['portfolio_score']
        # Domain composite
        df['overall_score'] = (df['tech_mean']*0.25 + df['soft_mean']*0.15 +
                                df['project_quality_score']*0.25 +
                                df['profile_composite']*0.10 +
                                df['interview_composite']*0.15 +
                                df['matched_skill']*0.10)
        return df

    train_df = derive(train_df); test_df = derive(test_df)
    train_cat = train_df.copy(); test_cat = test_df.copy()
    
    # Label encode (combined fit)
    for col in CAT_COLS + ['university_tier']:
        le = LabelEncoder()
        le.fit(pd.concat([train_df[col], test_df[col]]).astype(str))
        train_df[col] = le.transform(train_df[col].astype(str))
        test_df[col]  = le.transform(test_df[col].astype(str))
    
    drop_cols = ['student_id','mentor_feedback_text']
    if TARGET_COL in train_df.columns: drop_cols.append(TARGET_COL)
    train_df = train_df.drop(columns=drop_cols, errors='ignore')
    test_df  = test_df.drop(columns=drop_cols, errors='ignore')
    train_cat= train_cat.drop(columns=drop_cols, errors='ignore')
    test_cat = test_cat.drop(columns=drop_cols, errors='ignore')
    
    return train_df, test_df, train_cat, test_cat

t0 = time.time()
X_train, X_test, X_train_cat, X_test_cat = build_features(train, test)

# Birleştir: numerical + TF-IDF + BERT + Target encoding
parts_train = [X_train.reset_index(drop=True), train_nlp, train_te]
parts_test  = [X_test.reset_index(drop=True),  test_nlp,  test_te]
if BERT_OK:
    parts_train.append(train_bert)
    parts_test.append(test_bert)
X_train = pd.concat(parts_train, axis=1)
X_test  = pd.concat(parts_test, axis=1)

# Cat version (CatBoost native categorical)
parts_train_cat = [X_train_cat.reset_index(drop=True), train_nlp, train_te]
parts_test_cat  = [X_test_cat.reset_index(drop=True),  test_nlp,  test_te]
if BERT_OK:
    parts_train_cat.append(train_bert)
    parts_test_cat.append(test_bert)
X_train_cat = pd.concat(parts_train_cat, axis=1)
X_test_cat  = pd.concat(parts_test_cat, axis=1)

# Null doldurma
col_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(col_medians)
X_test  = X_test.fillna(col_medians)
for c in X_train_cat.columns:
    if pd.api.types.is_numeric_dtype(X_train_cat[c]) and X_train_cat[c].isna().any():
        X_train_cat[c] = X_train_cat[c].fillna(col_medians.get(c, 0))
        X_test_cat[c]  = X_test_cat[c].fillna(col_medians.get(c, 0))

print(f'\nFE süresi: {time.time()-t0:.1f}s')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
cat_features_idx = [X_train_cat.columns.get_loc(c) for c in CAT_COLS + ['university_tier']]

## 7. Optuna — Tüm Modeller İçin

ASHA pruner + erken durdurma ile verimli arama. **3-fold** (hızlı), final stacking 10-fold.

In [ ]:
# Hızlı 3-fold (sadece Optuna)
opt_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
OPT_SPLITS = list(opt_skf.split(X_train, y_bin))

# MLP ve KNN için standardize edilmiş features
scaler_global = StandardScaler()
X_train_std = pd.DataFrame(scaler_global.fit_transform(X_train), columns=X_train.columns)
X_test_std  = pd.DataFrame(scaler_global.transform(X_test),  columns=X_test.columns)

# Top features for Polynomial expansion (boyut patlamasını önler)
TOP_POLY = ['project_quality_score','technical_interview_score','problem_solving_score',
            'coding_score','cloud_score','matched_skill','role_composite','tech_mean',
            'soft_mean','overall_score','portfolio_score','profile_composite',
            'hr_interview_score','role_comp_minus_avg','matched_minus_tech']
X_train_poly = X_train[TOP_POLY].copy()
X_test_poly  = X_test[TOP_POLY].copy()
print(f'Standardized: {X_train_std.shape}, Poly base: {X_train_poly.shape}')

In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────────────
def xgb_objective(trial):
    params = {
        'n_estimators': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma': trial.suggest_float('gamma', 0, 2.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'early_stopping_rounds': 50, 'random_state': SEED, **xgb_gpu()
    }
    rmses = []
    for i, (tr, va) in enumerate(OPT_SPLITS):
        m = xgb.XGBRegressor(**params)
        m.fit(X_train.iloc[tr], y[tr], eval_set=[(X_train.iloc[va], y[va])], verbose=False)
        rmses.append(np.sqrt(mean_squared_error(y[va], m.predict(X_train.iloc[va]))))
        trial.report(np.mean(rmses), i)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(rmses)

print('XGBoost (100 trial)...')
xgb_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED),
                                 pruner=SuccessiveHalvingPruner())
t0 = time.time()
xgb_study.optimize(xgb_objective, n_trials=100, show_progress_bar=True)
print(f'XGB: {xgb_study.best_value:.4f} ({time.time()-t0:.0f}s)')

In [ ]:
# ── LightGBM ──────────────────────────────────────────────────────────────────
def lgb_objective(trial):
    params = {
        'n_estimators': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 400),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': SEED, 'verbose': -1, **lgb_gpu()
    }
    rmses = []
    for i, (tr, va) in enumerate(OPT_SPLITS):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_train.iloc[tr], y[tr], eval_set=[(X_train.iloc[va], y[va])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        rmses.append(np.sqrt(mean_squared_error(y[va], m.predict(X_train.iloc[va]))))
        trial.report(np.mean(rmses), i)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(rmses)

print('LightGBM (100 trial)...')
lgb_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED),
                                 pruner=SuccessiveHalvingPruner())
t0 = time.time()
lgb_study.optimize(lgb_objective, n_trials=100, show_progress_bar=True)
print(f'LGB: {lgb_study.best_value:.4f} ({time.time()-t0:.0f}s)')

In [ ]:
# ── CatBoost ──────────────────────────────────────────────────────────────────
def cat_objective(trial):
    params = {
        'iterations': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 100),
        'random_strength': trial.suggest_float('random_strength', 0.5, 5),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_seed': SEED, 'verbose': 0, 'loss_function': 'RMSE',
        'early_stopping_rounds': 50, **cat_gpu()
    }
    rmses = []
    for i, (tr, va) in enumerate(OPT_SPLITS):
        m = CatBoostRegressor(**params)
        m.fit(X_train_cat.iloc[tr], y[tr],
              eval_set=(X_train_cat.iloc[va], y[va]),
              cat_features=cat_features_idx, verbose=0)
        rmses.append(np.sqrt(mean_squared_error(y[va], m.predict(X_train_cat.iloc[va]))))
        trial.report(np.mean(rmses), i)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(rmses)

print('CatBoost (60 trial)...')
cat_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED),
                                 pruner=SuccessiveHalvingPruner())
t0 = time.time()
cat_study.optimize(cat_objective, n_trials=60, show_progress_bar=True)
print(f'CAT: {cat_study.best_value:.4f} ({time.time()-t0:.0f}s)')

In [ ]:
# ── HistGradientBoosting ──────────────────────────────────────────────────────
def hgb_objective(trial):
    params = {
        'max_iter': trial.suggest_int('max_iter', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 15, 300),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 100),
        'l2_regularization': trial.suggest_float('l2_regularization', 1e-3, 10, log=True),
        'random_state': SEED
    }
    rmses = []
    for tr, va in OPT_SPLITS:
        m = HistGradientBoostingRegressor(**params)
        m.fit(X_train.iloc[tr], y[tr])
        rmses.append(np.sqrt(mean_squared_error(y[va], m.predict(X_train.iloc[va]))))
    return np.mean(rmses)

print('HistGB (40 trial)...')
hgb_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED),
                                 pruner=MedianPruner(n_startup_trials=5))
t0 = time.time()
hgb_study.optimize(hgb_objective, n_trials=40, show_progress_bar=True)
print(f'HGB: {hgb_study.best_value:.4f} ({time.time()-t0:.0f}s)')

In [ ]:
# ── MLP (Neural — gerçek çeşitlilik) ──────────────────────────────────────────
MLP_ARCHS = [(128,), (256,), (128,64), (256,128), (256,128,64), (128,128,64)]

def mlp_objective(trial):
    arch_idx = trial.suggest_int('arch_idx', 0, len(MLP_ARCHS)-1)
    params = {
        'hidden_layer_sizes': MLP_ARCHS[arch_idx],
        'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
        'learning_rate_init': trial.suggest_float('lr_init', 1e-4, 5e-3, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [64, 128, 256]),
        'activation': 'relu',
        'solver': 'adam',
        'max_iter': 200,
        'early_stopping': True,
        'validation_fraction': 0.1,
        'n_iter_no_change': 15,
        'random_state': SEED
    }
    rmses = []
    for tr, va in OPT_SPLITS:
        m = MLPRegressor(**params)
        m.fit(X_train_std.iloc[tr], y[tr])
        rmses.append(np.sqrt(mean_squared_error(y[va], m.predict(X_train_std.iloc[va]))))
    return np.mean(rmses)

print('MLP (40 trial)...')
mlp_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED),
                                 pruner=MedianPruner(n_startup_trials=5))
t0 = time.time()
mlp_study.optimize(mlp_objective, n_trials=40, show_progress_bar=True)
print(f'MLP: {mlp_study.best_value:.4f} ({time.time()-t0:.0f}s)')

In [ ]:
# ── Ridge + PolynomialFeatures ────────────────────────────────────────────────
def ridge_objective(trial):
    params = {
        'alpha': trial.suggest_float('alpha', 0.1, 100, log=True),
        'degree': trial.suggest_int('degree', 2, 3),
    }
    rmses = []
    for tr, va in OPT_SPLITS:
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('poly', PolynomialFeatures(degree=params['degree'], interaction_only=False, include_bias=False)),
            ('ridge', Ridge(alpha=params['alpha'], random_state=SEED))
        ])
        pipe.fit(X_train_poly.iloc[tr], y[tr])
        rmses.append(np.sqrt(mean_squared_error(y[va], pipe.predict(X_train_poly.iloc[va]))))
    return np.mean(rmses)

print('Ridge+Poly (25 trial)...')
ridge_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
t0 = time.time()
ridge_study.optimize(ridge_objective, n_trials=25, show_progress_bar=True)
print(f'Ridge+Poly: {ridge_study.best_value:.4f} ({time.time()-t0:.0f}s)')

In [ ]:
# ── KNN (Distance-based, çeşitlilik için) ─────────────────────────────────────
def knn_objective(trial):
    params = {
        'n_neighbors': trial.suggest_int('n_neighbors', 10, 150),
        'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
        'p': trial.suggest_int('p', 1, 2),
    }
    rmses = []
    for tr, va in OPT_SPLITS:
        m = KNeighborsRegressor(**params, n_jobs=-1)
        m.fit(X_train_std.iloc[tr], y[tr])
        rmses.append(np.sqrt(mean_squared_error(y[va], m.predict(X_train_std.iloc[va]))))
    return np.mean(rmses)

print('KNN (20 trial)...')
knn_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
t0 = time.time()
knn_study.optimize(knn_objective, n_trials=20, show_progress_bar=True)
print(f'KNN: {knn_study.best_value:.4f} ({time.time()-t0:.0f}s)')

## 8. Stacking — 10-Fold × 3-Seed × 7 Model

In [ ]:
def make_oof(model_factory, splits, X_tr, X_te, y_, n_seeds=N_SEEDS, needs_es=False, lgb_es=False, cat=False, cat_idx=None):
    oof = np.zeros(len(y_)); test_p = np.zeros(len(X_te))
    for seed in range(n_seeds):
        fold_te = np.zeros(len(X_te))
        for tr_idx, val_idx in splits:
            m = model_factory(seed)
            if needs_es:
                m.fit(X_tr.iloc[tr_idx], y_[tr_idx],
                      eval_set=[(X_tr.iloc[val_idx], y_[val_idx])], verbose=False)
            elif lgb_es:
                m.fit(X_tr.iloc[tr_idx], y_[tr_idx],
                      eval_set=[(X_tr.iloc[val_idx], y_[val_idx])],
                      callbacks=[lgb.early_stopping(50, verbose=False)])
            elif cat:
                m.fit(X_tr.iloc[tr_idx], y_[tr_idx],
                      eval_set=(X_tr.iloc[val_idx], y_[val_idx]),
                      cat_features=cat_idx, verbose=0)
            else:
                m.fit(X_tr.iloc[tr_idx], y_[tr_idx])
            oof[val_idx] += m.predict(X_tr.iloc[val_idx]) / n_seeds
            fold_te += m.predict(X_te) / len(splits)
        test_p += fold_te / n_seeds
    return oof, test_p

print(f'Stacking ({N_FOLDS} fold × {N_SEEDS} seed × 7 model)...\n')

t0 = time.time()
xgb_factory = lambda s: xgb.XGBRegressor(**xgb_study.best_params, random_state=SEED+s,
    **xgb_gpu(), early_stopping_rounds=50, n_estimators=3000)
xgb_oof, xgb_test = make_oof(xgb_factory, FOLD_SPLITS, X_train, X_test, y, needs_es=True)
print(f'  XGB : {np.sqrt(mean_squared_error(y, xgb_oof)):.4f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
lgb_factory = lambda s: lgb.LGBMRegressor(**lgb_study.best_params, random_state=SEED+s,
    **lgb_gpu(), n_estimators=3000, verbose=-1)
lgb_oof, lgb_test = make_oof(lgb_factory, FOLD_SPLITS, X_train, X_test, y, lgb_es=True)
print(f'  LGB : {np.sqrt(mean_squared_error(y, lgb_oof)):.4f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
cat_factory = lambda s: CatBoostRegressor(**cat_study.best_params, random_seed=SEED+s,
    **cat_gpu(), iterations=3000, verbose=0, loss_function='RMSE', early_stopping_rounds=50)
cat_oof, cat_test = make_oof(cat_factory, FOLD_SPLITS, X_train_cat, X_test_cat, y, cat=True, cat_idx=cat_features_idx)
print(f'  CAT : {np.sqrt(mean_squared_error(y, cat_oof)):.4f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
hgb_factory = lambda s: HistGradientBoostingRegressor(**hgb_study.best_params, random_state=SEED+s)
hgb_oof, hgb_test = make_oof(hgb_factory, FOLD_SPLITS, X_train, X_test, y)
print(f'  HGB : {np.sqrt(mean_squared_error(y, hgb_oof)):.4f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
best_mlp = mlp_study.best_params
mlp_factory = lambda s: MLPRegressor(
    hidden_layer_sizes=MLP_ARCHS[best_mlp['arch_idx']],
    alpha=best_mlp['alpha'], learning_rate_init=best_mlp['lr_init'],
    batch_size=best_mlp['batch_size'], activation='relu', solver='adam',
    max_iter=300, early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=20, random_state=SEED+s)
mlp_oof, mlp_test = make_oof(mlp_factory, FOLD_SPLITS, X_train_std, X_test_std, y)
print(f'  MLP : {np.sqrt(mean_squared_error(y, mlp_oof)):.4f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
best_ridge = ridge_study.best_params
def ridge_factory(s):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=best_ridge['degree'], include_bias=False)),
        ('ridge', Ridge(alpha=best_ridge['alpha'], random_state=SEED+s))
    ])
ridge_oof, ridge_test = make_oof(ridge_factory, FOLD_SPLITS, X_train_poly, X_test_poly, y, n_seeds=1)
print(f'  Ridge+Poly : {np.sqrt(mean_squared_error(y, ridge_oof)):.4f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
def knn_factory(s):
    return KNeighborsRegressor(**knn_study.best_params, n_jobs=-1)
knn_oof, knn_test = make_oof(knn_factory, FOLD_SPLITS, X_train_std, X_test_std, y, n_seeds=1)
print(f'  KNN : {np.sqrt(mean_squared_error(y, knn_oof)):.4f}  ({time.time()-t0:.0f}s)')

oof_matrix  = np.column_stack([xgb_oof, lgb_oof, cat_oof, hgb_oof, mlp_oof, ridge_oof, knn_oof])
test_matrix = np.column_stack([xgb_test, lgb_test, cat_test, hgb_test, mlp_test, ridge_test, knn_test])
model_names = ['XGB','LGB','CAT','HGB','MLP','Ridge+Poly','KNN']

## 9. Meta-Learning — 2 Yöntem Karşılaştırması

In [ ]:
# Yöntem A: SLSQP — non-negative weight optimization
def weighted_rmse(w, preds, y_true):
    return np.sqrt(mean_squared_error(y_true, preds @ w))

n_models = oof_matrix.shape[1]
constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds = [(0.0, 1.0)] * n_models

best_slsqp_loss, best_slsqp_w = float('inf'), None
for trial in range(100):
    start = np.random.dirichlet(np.ones(n_models)) if trial > 0 else np.ones(n_models)/n_models
    res = minimize(weighted_rmse, start, args=(oof_matrix, y),
                    method='SLSQP', constraints=constraints, bounds=bounds,
                    options={'maxiter': 1000, 'ftol': 1e-10})
    if res.fun < best_slsqp_loss:
        best_slsqp_loss = res.fun; best_slsqp_w = res.x

# Yöntem B: Ridge meta-learner
from sklearn.model_selection import KFold as KF_simple
ridge_alphas = [0.01, 0.1, 1, 5, 10, 50, 100]
meta_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
best_ridge_alpha = None; best_ridge_loss = float('inf')
for alpha in ridge_alphas:
    fold_rmses = []
    for tr_idx, val_idx in meta_kf.split(oof_matrix, y_bin):
        meta = Ridge(alpha=alpha)
        meta.fit(oof_matrix[tr_idx], y[tr_idx])
        pred = meta.predict(oof_matrix[val_idx])
        fold_rmses.append(np.sqrt(mean_squared_error(y[val_idx], pred)))
    avg = np.mean(fold_rmses)
    if avg < best_ridge_loss:
        best_ridge_loss = avg; best_ridge_alpha = alpha

ridge_meta = Ridge(alpha=best_ridge_alpha)
ridge_meta.fit(oof_matrix, y)

# A vs B vs average
print(f'\n{"Yöntem":<20} | RMSE')
print('-'*40)
for n, w in zip(model_names, best_slsqp_w):
    print(f'  weight {n:<13}: {w:.3f}')
print(f'{"SLSQP weighted":<20} | {best_slsqp_loss:.4f}')
print(f'{"Ridge meta (α="+str(best_ridge_alpha)+")":<20} | {best_ridge_loss:.4f}')

# En iyiyi seç
if best_slsqp_loss < best_ridge_loss:
    final_oof  = oof_matrix @ best_slsqp_w
    final_test = test_matrix @ best_slsqp_w
    final_loss = best_slsqp_loss
    print('\n✓ SLSQP weighted seçildi')
else:
    final_oof  = ridge_meta.predict(oof_matrix)
    final_test = ridge_meta.predict(test_matrix)
    final_loss = best_ridge_loss
    print('\n✓ Ridge meta-learner seçildi')

# Bonus: 50/50 blend
blend_oof  = 0.5*(oof_matrix @ best_slsqp_w) + 0.5*ridge_meta.predict(oof_matrix)
blend_test = 0.5*(test_matrix @ best_slsqp_w) + 0.5*ridge_meta.predict(test_matrix)
blend_loss = np.sqrt(mean_squared_error(y, blend_oof))
print(f'{"50/50 Blend":<20} | {blend_loss:.4f}')
if blend_loss < final_loss:
    final_oof, final_test, final_loss = blend_oof, blend_test, blend_loss
    print('✓ Blend en iyi')

## 10. 🔄 Pseudo-Labeling A/B Test

In [ ]:
# Confidence: model tahminleri arası std
test_stds = test_matrix.std(axis=1)
PSEUDO_FRAC = 0.25
n_pseudo = int(len(test) * PSEUDO_FRAC)
conf_idx = np.argsort(test_stds)[:n_pseudo]
pseudo_labels = np.clip(final_test[conf_idx], 0, 100)

print(f'Pseudo-label: {n_pseudo} test örneği seçildi (en düşük std)')

# Augmented training set
X_train_aug     = pd.concat([X_train, X_test.iloc[conf_idx]], axis=0).reset_index(drop=True)
X_train_cat_aug = pd.concat([X_train_cat, X_test_cat.iloc[conf_idx]], axis=0).reset_index(drop=True)
X_train_std_aug = pd.concat([X_train_std, X_test_std.iloc[conf_idx]], axis=0).reset_index(drop=True)
X_train_poly_aug= pd.concat([X_train_poly, X_test_poly.iloc[conf_idx]], axis=0).reset_index(drop=True)
y_aug           = np.concatenate([y, pseudo_labels])

y_aug_bin = pd.qcut(y_aug, q=10, labels=False, duplicates='drop')
FOLD_SPLITS_AUG = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X_train_aug, y_aug_bin))

print(f'Augmented shape: {X_train_aug.shape} (orig: {X_train.shape[0]})')

In [ ]:
# Pseudo-augmented stacking
print('Pseudo-augmented stacking...\n')
n_orig = len(y)

t0 = time.time()
xgb_oof2, xgb_test2 = make_oof(xgb_factory, FOLD_SPLITS_AUG, X_train_aug, X_test, y_aug, needs_es=True)
lgb_oof2, lgb_test2 = make_oof(lgb_factory, FOLD_SPLITS_AUG, X_train_aug, X_test, y_aug, lgb_es=True)
cat_oof2, cat_test2 = make_oof(cat_factory, FOLD_SPLITS_AUG, X_train_cat_aug, X_test_cat, y_aug, cat=True, cat_idx=cat_features_idx)
hgb_oof2, hgb_test2 = make_oof(hgb_factory, FOLD_SPLITS_AUG, X_train_aug, X_test, y_aug)
mlp_oof2, mlp_test2 = make_oof(mlp_factory, FOLD_SPLITS_AUG, X_train_std_aug, X_test_std, y_aug)
ridge_oof2, ridge_test2 = make_oof(ridge_factory, FOLD_SPLITS_AUG, X_train_poly_aug, X_test_poly, y_aug, n_seeds=1)
knn_oof2, knn_test2 = make_oof(knn_factory, FOLD_SPLITS_AUG, X_train_std_aug, X_test_std, y_aug, n_seeds=1)

# Sadece original train için OOF değerlendir
oof_matrix2 = np.column_stack([xgb_oof2[:n_orig], lgb_oof2[:n_orig], cat_oof2[:n_orig],
                                hgb_oof2[:n_orig], mlp_oof2[:n_orig], ridge_oof2[:n_orig], knn_oof2[:n_orig]])
test_matrix2 = np.column_stack([xgb_test2, lgb_test2, cat_test2, hgb_test2, mlp_test2, ridge_test2, knn_test2])

print(f'Augmented stacking süresi: {time.time()-t0:.0f}s')
for i, n in enumerate(model_names):
    rmse_orig = np.sqrt(mean_squared_error(y, oof_matrix[:,i]))
    rmse_aug  = np.sqrt(mean_squared_error(y, oof_matrix2[:,i]))
    print(f'  {n:<12}: orig={rmse_orig:.4f} → aug={rmse_aug:.4f} ({rmse_orig-rmse_aug:+.4f})')

# Aug ensemble weight opt
best_slsqp_loss2 = float('inf'); best_slsqp_w2 = None
for trial in range(100):
    start = np.random.dirichlet(np.ones(n_models)) if trial > 0 else np.ones(n_models)/n_models
    res = minimize(weighted_rmse, start, args=(oof_matrix2, y),
                    method='SLSQP', constraints=constraints, bounds=bounds,
                    options={'maxiter': 1000, 'ftol': 1e-10})
    if res.fun < best_slsqp_loss2:
        best_slsqp_loss2 = res.fun; best_slsqp_w2 = res.x

print(f'\nAugmented ensemble RMSE: {best_slsqp_loss2:.4f}')
print(f'Original  ensemble RMSE: {final_loss:.4f}')

# En iyiyi seç
aug_test  = test_matrix2 @ best_slsqp_w2
aug_oof   = oof_matrix2  @ best_slsqp_w2
if best_slsqp_loss2 < final_loss:
    final_oof, final_test, final_loss = aug_oof, aug_test, best_slsqp_loss2
    print('✓ Pseudo-augmented ensemble seçildi')

# Bonus: orig + aug blend
blend2_oof  = 0.5*aug_oof + 0.5*final_oof
blend2_test = 0.5*aug_test + 0.5*final_test
blend2_loss = np.sqrt(mean_squared_error(y, blend2_oof))
print(f'\n50/50 Orig+Aug blend: {blend2_loss:.4f}')
if blend2_loss < final_loss:
    final_oof, final_test, final_loss = blend2_oof, blend2_test, blend2_loss
    print('✓ 50/50 blend seçildi')

final_test = np.clip(final_test, 0, 100)

## 11. Görselleştirme & Submission

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Model RMSE
rmses_indiv = [np.sqrt(mean_squared_error(y, oof_matrix[:,i])) for i in range(n_models)] + [final_loss]
names = model_names + ['FINAL']
colors = sns.color_palette('Set2', n_models+1)
axes[0,0].bar(names, rmses_indiv, color=colors)
for i, v in enumerate(rmses_indiv):
    axes[0,0].text(i, v+0.02, f'{v:.3f}', ha='center', fontsize=9)
axes[0,0].set_title('OOF RMSE — Model Karşılaştırması', fontsize=13)
axes[0,0].tick_params(axis='x', rotation=45)

axes[0,1].hist(y, bins=50, alpha=0.6, color='steelblue', label='Train', edgecolor='white')
axes[0,1].hist(final_test, bins=50, alpha=0.6, color='coral', label='Test (tahmin)', edgecolor='white')
axes[0,1].set_title('Dağılım'); axes[0,1].legend()

axes[1,0].scatter(y, final_oof, alpha=0.1, s=4, color='steelblue')
axes[1,0].plot([0,100],[0,100],'r--',lw=1)
axes[1,0].set_xlabel('Gerçek'); axes[1,0].set_ylabel('Tahmin')
axes[1,0].set_title('Ensemble OOF: Gerçek vs Tahmin')

axes[1,1].pie(best_slsqp_w, labels=model_names, autopct='%1.1f%%', colors=sns.color_palette('Set2', n_models))
axes[1,1].set_title('SLSQP Weight Distribution')

plt.tight_layout(); plt.show()

In [ ]:
submission = pd.DataFrame({
    'student_id'         : test_ids,
    'career_success_score': final_test
})
submission.to_csv('submission.csv', index=False)

final_mse = final_loss ** 2  # Yarışma metriği MSE
print('═' * 55)
print(f'  v4 Final OOF RMSE : {final_loss:.4f}')
print(f'  v4 Final OOF MSE  : {final_mse:.4f}  ← YARIŞMA SKORU (düşük=iyi)')
print('═' * 55)
print(f'\n  Hedef referans:')
print(f'    1. sıra (Berkay Fehmi Tekin): 83.42')
print(f'    10. sıra (CodeCosmos)        : 84.79')
print(f'    Senin v1 sonucun             : 89.54')
print(f'    v4 beklenen                  : {final_mse:.2f}')
print(f'\nsubmission.csv kaydedildi ✓ ({len(submission)} satır)')
print(submission.head(10))

---
## 📊 v4 Mimari Özet

| Katman | Açıklama |
|--------|----------|
| **Veri** | 10000 train + 10000 test, %16 null oranı |
| **Feature Engineering** | 200+ feature (numerical + role-skill + TF-IDF + BERT + TE) |
| **NLP** | TF-IDF (word+char) + Sentence-BERT (multilingual) |
| **Target Encoding** | Single + Cross-categorical, K-fold smoothed |
| **Base Models** | XGB, LGB, CAT, HGB (tree) + MLP (neural) + Ridge+Poly (linear) + KNN (distance) |
| **Hiperparametre Opt.** | Optuna TPE + ASHA pruner, 100+ trial |
| **CV** | 10-fold StratifiedKFold |
| **Multi-seed** | 3 seed averaging |
| **Meta-learner** | SLSQP + Ridge + Blend (en iyisi otomatik seçilir) |
| **Pseudo-labeling** | %25 confident, A/B test ile validate edilir |

### Beklenen Skor (MSE)
- v1 (mevcut): 89.54
- 1. sıra: 83.42
- **v4 hedef: ~72-78** (ilk 5-10)

### Eğer Hâlâ Yeterli Değilse
- **Daha büyük BERT modeli** (multilingual-e5-large)
- **TabNet** veya **FT-Transformer** neural base
- **2 round pseudo-labeling**
- **Daha çok seed** (5-10)